# Cycle 2 — Tuning (Chronological Split)

Same `RandomizedSearchCV` configuration as `notebooks/cycle2_tuning.ipynb`. Only the train/test split is chronological (by `matchId`). Inner CV is still 5-fold stratified on the time-ordered training partition.

## Setup & data (chronological by matchId)

In [2]:
import json, math, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier

with open('../../../data/raw/events_England.json') as f: raw = json.load(f)
df = pd.DataFrame(raw)
shots = df[df['eventName']=='Shot'].copy().reset_index(drop=True)

shots['Goal']       = shots['tags'].apply(lambda t: int(any(x['id']==101 for x in t)))
shots['Left_Foot']  = shots['tags'].apply(lambda t: int(any(x['id']==401 for x in t)))
shots['Right_Foot'] = shots['tags'].apply(lambda t: int(any(x['id']==402 for x in t)))
shots['Header']     = shots['tags'].apply(lambda t: int(any(x['id']==403 for x in t)))
shots = shots[shots['matchPeriod'].isin(['1H','2H'])].copy().reset_index(drop=True)
shots['First_Half'] = (shots['matchPeriod']=='1H').astype(int)
shots['X'] = shots['positions'].apply(lambda p: p[0]['x'])
shots['Y'] = shots['positions'].apply(lambda p: p[0]['y'])

PITCH_L, PITCH_W = 105.0, 68.0
POST_L, POST_R   = 30.34, 37.66
GOAL_Y = (POST_L+POST_R)/2
x_m = shots['X']/100.0*PITCH_L; y_m = shots['Y']/100.0*PITCH_W
shots['Distance'] = np.sqrt((x_m-PITCH_L)**2+(y_m-GOAL_Y)**2)
dx = PITCH_L-x_m
shots['Angle'] = np.abs(np.degrees(np.arctan2(POST_R-y_m,dx)-np.arctan2(POST_L-y_m,dx)))
shots['Player_Rank'] = 0.0
proc = pd.read_csv('../../../data/processed/wyscout_shots_processed.csv')
if 'Player_Rank' in proc.columns and len(proc)==len(shots):
    shots['Player_Rank'] = proc['Player_Rank'].values

FEATURES = ['X','Y','Distance','Angle','Left_Foot','Right_Foot','Header','First_Half','Player_Rank']
shots = shots[['matchId','Goal']+FEATURES].dropna().sort_values('matchId').reset_index(drop=True)

ids = shots['matchId'].drop_duplicates().sort_values().tolist()
boundary = ids[int(len(ids)*0.8)]
train = shots['matchId'] < boundary
X_train, y_train = shots.loc[train, FEATURES], shots.loc[train,'Goal']
X_test,  y_test  = shots.loc[~train,FEATURES], shots.loc[~train,'Goal']

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

spw = (y_train==0).sum()/(y_train==1).sum()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,} | scale_pos_weight: {spw:.2f}')

Train: 6,818 | Test: 1,633 | scale_pos_weight: 8.30


## XGBoost search

In [3]:
xgb_param_grid = {
    'n_estimators':     [100,200,300,500],
    'max_depth':        [3,4,5,6],
    'learning_rate':    [0.01,0.05,0.1,0.2],
    'subsample':        [0.7,0.8,1.0],
    'colsample_bytree': [0.7,0.8,1.0],
    'min_child_weight': [1,3,5],
    'gamma':            [0,0.1,0.2],
    'scale_pos_weight': [spw, spw*0.5, spw*1.5],
}

xgb = XGBClassifier(random_state=42, eval_metric='auc', verbosity=0)
search_xgb = RandomizedSearchCV(xgb, xgb_param_grid, n_iter=50, cv=cv,
                                 scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1)
search_xgb.fit(X_train_s, y_train)

y_prob_xgb_t = search_xgb.best_estimator_.predict_proba(X_test_s)[:,1]
y_pred_xgb_t = search_xgb.best_estimator_.predict(X_test_s)
print('Best params:', search_xgb.best_params_)
print(f'Best CV AUC : {search_xgb.best_score_:.4f}')
print(f'Test AUC    : {roc_auc_score(y_test, y_prob_xgb_t):.4f}')
print(f'Test Acc    : {accuracy_score(y_test, y_pred_xgb_t)*100:.2f}%')
print(classification_report(y_test, y_pred_xgb_t, target_names=['No Goal','Goal']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.7, 'scale_pos_weight': 8.30150068212824, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV AUC : 0.8083
Test AUC    : 0.8342
Test Acc    : 72.44%
              precision    recall  f1-score   support

     No Goal       0.96      0.72      0.82      1452
        Goal       0.26      0.78      0.39       181

    accuracy                           0.72      1633
   macro avg       0.61      0.75      0.60      1633
weighted avg       0.89      0.72      0.77      1633



## Random Forest search

In [4]:
rf_param_grid = {
    'n_estimators':      [100,200,300,500],
    'max_depth':         [None,5,10,15,20],
    'min_samples_split': [2,5,10],
    'min_samples_leaf':  [1,2,4],
    'max_features':      ['sqrt','log2',None],
    'class_weight':      ['balanced','balanced_subsample'],
}

rf = RandomForestClassifier(random_state=42)
search_rf = RandomizedSearchCV(rf, rf_param_grid, n_iter=50, cv=cv,
                                scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1)
search_rf.fit(X_train_s, y_train)

y_prob_rf_t = search_rf.best_estimator_.predict_proba(X_test_s)[:,1]
print('Best params:', search_rf.best_params_)
print(f'Best CV AUC : {search_rf.best_score_:.4f}')
print(f'Test AUC    : {roc_auc_score(y_test, y_prob_rf_t):.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 5, 'class_weight': 'balanced'}
Best CV AUC : 0.8067
Test AUC    : 0.8292


## Side-by-side comparison

Random-split tuned numbers from `notebooks/cycle2_tuning.ipynb`:
- XGBoost Tuned: AUC 0.8183
- Random Forest Tuned: AUC 0.8176

In [5]:
comp = pd.DataFrame([
    {'Model':'XGBoost Tuned',       'Chrono AUC':roc_auc_score(y_test,y_prob_xgb_t),'Random AUC':0.8183},
    {'Model':'Random Forest Tuned', 'Chrono AUC':roc_auc_score(y_test,y_prob_rf_t), 'Random AUC':0.8176},
])
comp['Delta']     = (comp['Chrono AUC']-comp['Random AUC']).round(4)
comp['Chrono AUC']= comp['Chrono AUC'].round(4)
print(comp.to_string(index=False))

              Model  Chrono AUC  Random AUC  Delta
      XGBoost Tuned      0.8342      0.8183 0.0159
Random Forest Tuned      0.8292      0.7800 0.0492


### Observations:

- XGBoos tuned chrono achived the highest **AUC** compared to tuned XGBoost on random splits that's illustrate the effictifness of chronological split compared to random split.

## Save best chronological model

Picks whichever tuned model has the highest test AUC and overwrites the deployed `models/cycle2_*` artefacts so the API serves the chronologically-trained model.

In [6]:
import joblib, os

xgb_test_auc = roc_auc_score(y_test, y_prob_xgb_t)
rf_test_auc  = roc_auc_score(y_test, y_prob_rf_t)

if xgb_test_auc >= rf_test_auc:
    best_model = search_xgb.best_estimator_
    best_name  = 'XGBoost Tuned (chronological)'
    best_auc   = xgb_test_auc
else:
    best_model = search_rf.best_estimator_
    best_name  = 'Random Forest Tuned (chronological)'
    best_auc   = rf_test_auc

print(f'Saving: {best_name}  (test AUC = {best_auc:.4f})')

os.makedirs('../../../models', exist_ok=True)
joblib.dump(best_model,            '../../../models/cycle2_best_model.pkl')
joblib.dump(scaler,                '../../../models/cycle2_scaler.pkl')
joblib.dump(list(X_train.columns), '../../../models/cycle2_feature_cols.pkl')
print('Saved → ../../../models/cycle2_best_model.pkl')
print('Saved → ../../../models/cycle2_scaler.pkl')
print('Saved → ../../../models/cycle2_feature_cols.pkl')

Saving: XGBoost Tuned (chronological)  (test AUC = 0.8342)
Saved → ../models/cycle2_best_model.pkl
Saved → ../models/cycle2_scaler.pkl
Saved → ../models/cycle2_feature_cols.pkl
